<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter9/Advanced_Interface_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Interface features

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


## Using state to persist data ##

Gradio supports session state, where data persists across multiple submits within a page load. Session state is useful for building demos of, for example, chatbots where we want to persist data as the user interacts with the model. Note that session state does not share data between users of our model.

To store data in a session state, we need to do three things:

1. Pass an extra parameter into the function, which represents the state of the interface.
2. At the end of the function, return the updated value of the state as an extra return value.
3. Add the 'state' input and 'state' output components when creating our Interface

A chatbot example would be:

In [5]:
import random

import gradio as gr

def chat(message, history):
  history = history or []
  if message.startswith("How many"):
    response = random.randint(1, 10)
  elif message.startswith("How"):
    response = random.choice(['Great', 'Good', 'Okay', 'Bad'])
  elif message.startswith("Where"):
    response = random.choice(['Here', 'There', 'Somewhere'])
  else:
    response = "I don't know"
  history.append((message, response))
  return history, history

gr.ChatInterface(fn=chat).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2155e2df5d35825bf4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The state of the output components presists across submits. Note: we can pass in a default value to the state parameter, which is used as the initial value of the state.

## Using interpretation to understand predictions ##

Most ML models are black boxes and the internal logic of the function is hidden from the end user. It is easy to add interpretation to the model by simply setting the interpretation keyword in the Interface class to default. This allows the users to understand what parts of the input are responsible for the output. Let's take a look at the simple interface below which shows an image classifier that also includes interpretation:

In [8]:
import requests
import tensorflow as tf

import gradio as gr

inception_net = tf.keras.applications.MobileNetV2()

# Download human-readable labels for ImageNet.
response = requests.get("https://git.io/JJkYN")
labels = response.text.split("\n")

def classify_image(inp):
  inp = inp.reshape((-1, 224, 224, 3))
  inp = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
  prediction = inception_net.predict(inp).flatten()
  return {labels[i]: float(prediction[i]) for i in range(1000)}

image = gr.Image()
label = gr.Label(num_top_classes=3)

title = "Gradio Image Classification + Interpretation Example"
gr.Interface(fn=classify_image, inputs=image, outputs=label, title=title).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://15532ec09777a29530.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Besides the default interpretation method Gradio provides, we can also specify shap for the interpretation parameter and set the num_shap parameter. This uses Shapley-based interpretation.

As we've seen, this class makes it simple to create machine learning demos in a few lines of code. However, sometimes we'll want to customise our demo by changing the laylout or chaining multiple prediction functions together. It would be nice if we could somehow split the Interface into customizable "blocks". We'll see that in the next section!